# 07 Pairwise Similarity: gpt-4o vs gpt-5.1

Uses `data/comparison-pairs.jsonl` — five hand-crafted question/answer pairs, one response per model per row.

`SimilarityEvaluator` is called with:
- `response` = gpt-5.1 answer
- `ground_truth` = gpt-4o answer

A score of 1 means the answers are very different; 5 means they are nearly identical in wording. The evaluator model is gpt-4o (`chat4o`).

## Load environment

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AOAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AOAI_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AOAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')
AOAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')

print('Environment loaded.')
print(f'  endpoint   : {AOAI_ENDPOINT}')
print(f'  deployment : {AOAI_DEPLOYMENT}')
print(f'  api_version: {AOAI_API_VERSION}')

Environment loaded.
  endpoint   : https://aoaievalgw021uks.openai.azure.com
  deployment : chat4o


## Run SimilarityEvaluator

In [33]:
from azure.ai.evaluation import SimilarityEvaluator, RelevanceEvaluator

pairs_path = Path('../data/comparison-pairs.jsonl')
# utf-8-sig handles BOM that Windows tools may add
pairs = [json.loads(line) for line in pairs_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
print(f'Loaded {len(pairs)} comparison pairs')

model_config = {
    'azure_endpoint': AOAI_ENDPOINT,
    'api_key': AOAI_KEY,
    'azure_deployment': AOAI_DEPLOYMENT,
    'api_version': '2024-08-01-preview',
}

similarity_evaluator = SimilarityEvaluator(model_config=model_config)
relevance_evaluator = RelevanceEvaluator(model_config=model_config)

similarity_rows = []

for i, pair in enumerate(pairs, start=1):
    query = pair['query']
    response_4o = pair['response_4o']
    response_51 = pair['response_51']

    # Similarity score: gpt-4o answer as ground truth, gpt-5.1 answer as prediction
    sim_result = similarity_evaluator(
        query=query,
        response=response_51,
        ground_truth=response_4o,
    )
    similarity_score = sim_result.get('similarity', None)

    # Evaluator-native rationale
    rel_result = relevance_evaluator(
        query=query,
        response=response_51,
        context=response_4o,
    )
    evaluator_reason = rel_result.get('relevance_reason', '')

    similarity_rows.append({
        'pair': i,
        'query': query,
        'similarity': similarity_score,
        'evaluator_reason': evaluator_reason,
        'response_4o': response_4o,
        'response_51': response_51,
    })

    print(f"Pair {i}  similarity={similarity_score}  query={query[:60]!r}")

print(f'\nDone - {len(similarity_rows)} pairs scored.')

Loaded 5 comparison pairs
Pair 1  similarity=1.0  query='Should I put HTTP retry logic in my service code or delegate'
Pair 2  similarity=4.0  query='Is it better to use async or sync code for a Python web scra'
Pair 3  similarity=4.0  query='Write a short story about a student who finds confidence bef'
Pair 4  similarity=3.0  query='Is Python a good choice for a high-performance API server?'
Pair 5  similarity=5.0  query='What is the difference between authentication and authorisat'

Done - 5 pairs scored.


## Display results

In [34]:
from IPython.display import Markdown, display
import importlib
import sys

app_dir = Path('../app').resolve()
if str(app_dir) not in sys.path:
    sys.path.append(str(app_dir))

import comparison_renderer
importlib.reload(comparison_renderer)

# Notebook 05-style full rendering: no truncation of model responses
render_rows = []
for row in similarity_rows:
    render_rows.append({
        'scenario_id': f"pair_{row['pair']:02d}",
        'category': 'Pairwise Similarity',
        'system_label': 'Similarity score',
        'system': f"{row['similarity']}",
        'user_label': 'Prompt',
        'user': row['query'],
        'difference_label': 'Evaluator reasoning',
        'difference_explanation': row.get('evaluator_reason', ''),
        'primary_deployment': AOAI_DEPLOYMENT,
        'secondary_deployment': 'chat51',
        'primary_status': 'success',
        'secondary_status': 'success',
        'primary_response': row['response_4o'],
        'secondary_response': row['response_51'],
    })

comparison_renderer.render_comparison(render_rows)

avg = sum(r['similarity'] for r in similarity_rows if isinstance(r['similarity'], (int, float))) / len(similarity_rows)
display(Markdown(f"Average similarity across {len(similarity_rows)} pairs: **{avg:.2f}**"))

# Save results
output_path = Path('../outputs/similarity-pairs-results.json')
output_path.write_text(json.dumps(similarity_rows, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved to {output_path}')

gpt-4o - chat4o,gpt-5.1 - chat51
"Put retry logic in your service code. You have full control over which status codes and exception types are worth retrying, you can log intermediate attempts, and you avoid the operational overhead of running a service mesh for a single service. A short try/sleep loop or the tenacity library covers most cases.","Delegate retries to the service mesh (Istio or Linkerd) rather than duplicating the logic in every service. Retries are an infrastructure concern: centralising them in the mesh means consistent behaviour across all services, configurable via YAML without a redeploy, and built-in observability and circuit breaking at no extra code cost. If you are already on Kubernetes, this is the right layer for it."
gpt-4o - chat4o,gpt-5.1 - chat51
"For scraping many URLs, async is significantly faster. I/O wait dominates in scraping, so asyncio with aiohttp or httpx lets you fire hundreds of requests concurrently without spawning hundreds of threads. Use sync requests only for a handful of URLs or a quick one-off script.","It depends on scale. For fewer than roughly 20 URLs, sync requests with a ThreadPoolExecutor is simpler and fast enough. For hundreds of URLs, asyncio with aiohttp or httpx offers better throughput. For large crawls, Scrapy manages concurrency, politeness delays, and pipelines for you so you don't have to wire up the event loop manually. Async raw code is not always the right answer — the added complexity is only worth it when you have the URL volume to justify it."
gpt-4o - chat4o,gpt-5.1 - chat51
"Mia stood at the back of Room 12, palms damp around her note cards. Every laugh from the hallway sounded like a warning. She had practiced at home, but now the words felt slippery. When her name was called, she walked to the front and saw her friend Theo in the first row, giving her a small thumbs-up. Mia took one breath, then another. She opened with the story of her grandmother teaching her to grow tomatoes on a tiny balcony. Something shifted. The class leaned in. Her voice steadied. By the last slide, she was speaking without looking down. When she finished, there was a beat of silence and then applause. Mia sat down smiling, surprised to realize she wanted to present again.","The projector hummed like a nervous insect while Leila waited for her turn. Her presentation on urban gardens was next, and she had already imagined three disasters: forgetting every line, dropping her flash drive, and fainting in front of twenty-seven classmates. She pressed her fingers to the edge of her notebook and remembered what her older brother had told her that morning: 'You do not need to be fearless. You just need your first sentence.' When the teacher called her name, Leila walked to the front, knees unsteady, and clicked to the title slide. 'Last summer, a tomato plant grew through a crack in our apartment parking lot,' she began. A few students looked up. She continued, describing how something fragile could thrive in hard places. By slide three she stopped reading her notes. By slide five she was answering questions before they were fully asked. She compared rooftop gardens to open-air classrooms, talked about food deserts, and explained why planting herbs outside the library had changed how younger students used the space. At the end, the room was quiet in the good way - attentive, not awkward. Then came claps, then more claps, and even the students who usually stared at their phones were looking at her. Back at her desk, Leila wrote a single line in the margin of her notebook: 'Confidence is what shows up after you start.'"
gpt-4o - chat4o,gpt-5.1 - chat51
"Yes, for most APIs Python is fast enough. Async frameworks like FastAPI handle high concurrency well, and the bottleneck is usually the database or network, not the Python runtime. Profile before assuming Python is too slow — premature optimisation leads to unnecessary complexity.","Python has real performance limits for high-throughput sc

Average similarity across 5 pairs: **3.40**

Saved to ..\outputs\similarity-pairs-results.json
